In [ ]:
# Install necessary libraries
!pip install shap transformers datasets torch scikit-learn

# Import libraries
import torch
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from transformers.data.data_collator import DataCollatorWithPadding
from datasets import Dataset
import numpy as np # Import numpy and assign it to the alias 'np'

# Step 1: Upload and Load Dataset
# from google.colab import files
# uploaded = files.upload()  # Upload the dataset file
# DATA_FILE = list(uploaded.keys())[0]  # Get the uploaded file name
DATA = pd.read_csv("DefaktS_Twitter.csv")
DATA['label'] = DATA['binary_label'].astype(int)
DATA['text'] = DATA['text'].str.replace(r"https:\/\/t.co\/\S+", "[URL]", regex=True)

# Step 2: Balance the dataset
minority_class = DATA[DATA['label'] == 1]
majority_class = DATA[DATA['label'] == 0]
oversampled_minority = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=42
)
balanced_data = pd.concat([majority_class, oversampled_minority])

# Step 3: Train-Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced_data['text'], balanced_data['label'], test_size=0.2, stratify=balanced_data['label'], random_state=42
)

# Step 4: Compute Class Weights
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_labels), y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# Step 5: Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Step 6: Tokenizer and Base Model
#MODEL_NAME = "distilbert-base-uncased"
MODEL_NAME = "distilbert-base-german-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Custom model class to include class weights and dropout
class WeightedBERT(torch.nn.Module):
    def __init__(self, model, class_weights):
        super(WeightedBERT, self).__init__()
        self.model = model
        self.dropout = torch.nn.Dropout(0.4)  # Increased dropout for regularization
        self.register_buffer("class_weights", class_weights)

    def forward(self, input_ids, attention_mask, labels=None):
        device = input_ids.device
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels, weight=self.class_weights)
        return {"loss": loss, "logits": logits}

# Load Pretrained BERT Model and Wrap It
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = WeightedBERT(base_model, class_weights)

# Step 7: Tokenize the Dataset
def tokenize_function(sample):
    return tokenizer(sample['text'], truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Step 8: Data Collator for Dynamic Padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 9: Define Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Step 10: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.2,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    lr_scheduler_type="cosine_with_min_lr",
    warmup_steps=500,
    lr_scheduler_kwargs={"min_lr": 1e-6},
    report_to=[],
)

# Step 11: Initialize Trainer with Early Stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Step 12: Train the Model
trainer.train()

# # Step 13: Save the Model
# model.save_pretrained("./saved_model")
# tokenizer.save_pretrained("./saved_model")

# Save the base Hugging Face model (inside WeightedBERT)
model.model.save_pretrained("./saved_model")  # Save the underlying Hugging Face model
tokenizer.save_pretrained("./saved_model")  # Save the tokenizer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

...

Map:   0%|          | 0/18833 [00:00<?, ? examples/s]

Map:   0%|          | 0/4709 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.438800,0.306295,0.867488,0.868742,0.860417,0.877230
2,0.271300,0.309257,0.887874,0.883956,0.915756,0.854291
3,0.277000,0.320376,0.908048,0.908553,0.903402,0.913764
4,0.221100,0.425072,0.913357,0.912821,0.918315,0.907392
5,0.196400,0.466174,0.911446,0.911746,0.908477,0.915038


('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json',
 './saved_model/tokenizer.json')